# casmi encoder C2-PC evaluation (CPU)
Generated by `scripts/kaggle_encoder_kernel.py`.

In [ ]:
# ---- settings (generated by scripts/kaggle_encoder_kernel.py at f3963cddc2fc) ----
from pathlib import Path
OWNER = 'ograhulrajkumar'
TRAIN_KERNEL = 'casmi-encoder-train'       # its output (encoder_<arm>.pt) is attached as a data source
ARMS = ['bce', 'focal_con']
MAX_MOL = 40               # 0 = all C2-PC molecules
BUDGET_MIN = 40.0         # stop after this many minutes (hash order = unbiased subsample)
SHARD, N_SHARDS = 0, 1
CODE_DATASET = 'casmi-encoder-code'
BUNDLE_DATASET = 'casmi-encoder-bundle'
RDKIT_WHEEL_DATASET = 'metric/rdkit-2026-3-3-wheel'
WORK = Path("/kaggle/working")

In [ ]:
import os, sys, time, json, glob, shutil, hashlib, subprocess, traceback
from pathlib import Path

INPUT = Path("/kaggle/input")


def find_dataset(slug, owner=None):
    """Mount directory of an attached dataset, found by slug (any Kaggle mount layout)."""
    for cand in ([INPUT / "datasets" / owner / slug] if owner else []) + [INPUT / slug]:
        if cand.is_dir():
            return cand
    hits = []
    for dirpath, dirnames, _ in os.walk(INPUT):
        depth = len(Path(dirpath).relative_to(INPUT).parts)
        if depth >= 4:
            dirnames[:] = []
            continue
        for d in sorted(dirnames):
            if d.lower() == slug.lower():
                hits.append(Path(dirpath) / d)
    if owner:
        hits.sort(key=lambda p: (owner not in p.parts, len(p.parts), str(p)))
    return hits[0] if hits else None


def find_test(competition):
    """The competition's test.parquet (replaced by the hidden test on a scoring rerun)."""
    comp = find_dataset(competition)
    if comp is not None:
        hits = sorted(comp.rglob("test.parquet"))
        if hits:
            return hits[0]
    hits = sorted(INPUT.rglob("test.parquet"))
    if not hits:
        raise FileNotFoundError("no test.parquet under /kaggle/input")
    return hits[0]

In [ ]:
T0 = time.time()
try:
    import rdkit  # noqa: F401
except ImportError:
    owner, slug = RDKIT_WHEEL_DATASET.split("/")
    wd = find_dataset(slug, owner)
    r = subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--no-index", "--quiet",
                        f"--find-links={wd}", "rdkit==2026.3.3"], capture_output=True, text=True)
    print("rdkit install", r.returncode, r.stderr[-500:])
CODE = find_dataset(CODE_DATASET, OWNER)
BUNDLE = find_dataset(BUNDLE_DATASET, OWNER)
NPDIR = sorted(p.parent for p in INPUT.rglob("np_fp.npy"))[0]
PCDIR = sorted(p.parent for p in INPUT.rglob("mass_*.parquet"))[0]
models = []
for arm in ARMS:
    hits = sorted(INPUT.rglob(f"encoder_{arm}.pt"))
    print(arm, hits)
    if hits:
        models += ["--model", f"{arm}={hits[0]}"]
print("code", CODE, "bundle", BUNDLE, "np", NPDIR, "pubchem", PCDIR)
env = dict(os.environ, PYTHONPATH=str(CODE), CASMI_ROOT=str(WORK / "casmi_run"),
           NUMBA_CACHE_DIR=str(WORK / "numba_cache"), PYTHONUNBUFFERED="1")
cmd = [sys.executable, str(CODE / "scripts" / "eval_encoder_pc.py"), "--bundle", str(BUNDLE),
       "--np-dir", str(NPDIR), "--pubchem-dir", str(PCDIR), "--n-jobs", "4", "--threads", "4",
       "--max-mol", str(MAX_MOL), "--budget-min", str(BUDGET_MIN), "--shard", str(SHARD),
       "--n-shards", str(N_SHARDS), "--out", str(WORK / "c2pc.json")] + models
import torch
if torch.cuda.is_available():
    cmd += ["--device", "cuda"]
r = subprocess.run(cmd, env=env, capture_output=True, text=True)
print(r.stdout[-3000:])
print(r.stderr[-6000:])
if (WORK / "c2pc.json").exists():
    print((WORK / "c2pc.json").read_text())
print(f"total {(time.time() - T0) / 60:.1f} min")